# Music Instrument Classification - Complete Implementation

This notebook implements both the baseline SVM model and the Attention-based model for music instrument classification. The structure follows the guidelines outlined in Project 6, with integration of annotations for multi-label classification. Feature extraction includes MFCC, spectral contrast, and zero-crossing rate, along with error analysis and additional data segmentation steps.

## Phase 1: Import Libraries and Paths

1. Import necessary libraries (librosa, numpy, PyTorch, sklearn).
2. Define dataset and annotations paths.

In [21]:
# Import Libraries
import os
from tqdm import tqdm
import librosa
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import glob
from collections import defaultdict
import csv

# Definizione dei percorsi (modifica se necessario)
annotations_path = "D:/MedleyDB/MedleyDB/Annotations"
dataset_path = "D:/MedleyDB/MedleyDB/Audio"  # o il path giusto dei tuoi file audio


## Phase 2: Data Loading and Annotations

1. Load audio files and normalize them.
2. Load and process annotations for multi-label classification.

In [22]:
def load_instrument_annotations_with_times():
    # Dizionario finale
    activations = defaultdict(list)

    # Percorso delle annotazioni
    sourceid_path = os.path.join(annotations_path, "Instrument_Activations", "SOURCEID")
    source_files = glob.glob(os.path.join(sourceid_path, "*.lab"))
    print(f"Trovati {len(source_files)} file SOURCEID")

    for filepath in source_files:
        filename = os.path.basename(filepath)
        track_name = filename.replace("_SOURCEID.lab", "")

        with open(filepath, 'r') as f:
            reader = csv.reader(f)
            header = next(reader, None)  # Salta la prima riga

            for row in reader:
                if len(row) >= 3:
                    start_time = float(row[0].strip())
                    end_time = float(row[1].strip())
                    instrument = row[2].strip().lower()

                    # Salva la tripletta (start, end, instrument)
                    activations[track_name].append((start_time, end_time, instrument))

    return activations

# === Caricamento delle annotazioni ===
instrument_annotations = load_instrument_annotations_with_times()
print(f"\n✅ Caricate {len(instrument_annotations)} tracce con annotazioni degli strumenti")

# === Caricamento dei file audio ===
audio_files = glob.glob(os.path.join(dataset_path, '**', '*.wav'), recursive=True)
print(f"✅ Trovati {len(audio_files)} file audio")

# === Debug: stampa esempi
print("\n🎵 Esempi di annotazioni caricate:")
for i, (track, segments) in enumerate(instrument_annotations.items()):
    print(f"{track}: {segments[:3]} ...")  # mostra i primi 3 segmenti con tempi
    if i == 2:
        break

Trovati 122 file SOURCEID

✅ Caricate 122 tracce con annotazioni degli strumenti
✅ Trovati 2879 file audio

🎵 Esempi di annotazioni caricate:
AClassicEducation_NightOwl: [(0.743, 18.9475, 'electric bass'), (19.8763, 37.895, 'electric bass'), (38.9631, 57.0746, 'electric bass')] ...
AimeeNorwich_Child: [(0.418, 160.8678, 'drum set'), (12.0279, 174.4283, 'electric bass'), (20.3871, 161.1465, 'clean electric guitar')] ...
AimeeNorwich_Flying: [(5.4799, 188.964, 'drum set'), (77.3224, 84.3349, 'fx/processed sound'), (7.9412, 77.8333, 'electric bass')] ...


We decided to take only the instrument annotations in the annotations since they are the most relevant for our classification problem

## Phase 3: Data Normalization and Segmentation

1. Normalize audio amplitude.
2. Create segments of fixed duration.
3. Assign multi-labels to each segment using annotations.

In [26]:
def normalize_audio(audio):
    max_val = np.max(np.abs(audio))
    if max_val == 0:
        return audio  # Segmento silenzioso → lo lasci così com'è
    return audio / max_val


def segment_audio(audio, sr, segment_length=3):
    segment_samples = segment_length * sr
    return [audio[i:i + segment_samples] for i in range(0, len(audio), segment_samples)]

def get_instruments(track_id, instrument_annotations):
    if track_id in instrument_annotations:
        return list(instrument_annotations[track_id])
    return []

import re

def clean_track_name(name):
    return re.sub('(_RAW.*|_MIX|_STEM.*)', '', name)


In [ ]:
segment_length = 3  # secondi
sr_target = 22050

# Liste finali
segments_list = []
labels_list = []
track_list = []

for audio_file in tqdm(audio_files, desc="Processing audio files"):
    raw_track_name = os.path.basename(audio_file).replace('.wav', '')
    track_name = clean_track_name(raw_track_name)

    if track_name not in instrument_annotations:
        continue

    try:
        audio, sr = librosa.load(audio_file, sr=sr_target)
    except Exception:
        continue

    # Itera su tutti i periodi di attività
    for start_time, end_time, instrument in instrument_annotations[track_name]:
        duration = end_time - start_time
        if duration < segment_length:
            continue

        try:
            segment_audio, _ = librosa.load(audio_file, sr=sr_target, offset=start_time, duration=segment_length)
        except Exception:
            continue

        if len(segment_audio) < segment_length * sr_target:
            continue

        max_val = np.max(np.abs(segment_audio))
        if max_val > 0:
            segment_audio = segment_audio / max_val

        segments_list.append(segment_audio)
        labels_list.append(instrument)
        track_list.append(track_name)

print(f"\n✅ Segmenti totali creati: {len(segments_list)}")

Processing audio files:  56%|█████▌    | 1617/2879 [23:13<30:53,  1.47s/it]  

## Phase 4: Feature Extraction

1. Extract MFCC, Spectral Contrast, and Zero Crossing Rate.
2. Implement Co-occurrence Matrix for instrument interactions.

In [ ]:

def extract_features(audio, sr):
    # MFCC
    mfcc = librosa.feature.mfcc(audio, sr=sr, n_mfcc=13).flatten()
    # Spectral Contrast
    spectral_contrast = librosa.feature.spectral_contrast(audio, sr=sr).flatten()
    # Zero Crossing Rate
    zcr = librosa.feature.zero_crossing_rate(audio).flatten()
    # Concatena tutte le feature
    return np.concatenate([mfcc, spectral_contrast, zcr])

  0%|          | 0/2879 [00:07<?, ?it/s]


TypeError: mfcc() takes 0 positional arguments but 1 positional argument (and 2 keyword-only arguments) were given

## Phase 5: SVM Baseline Implementation

Train a baseline SVM model using MFCC and spectral contrast features.


In [ ]:
# SVM Baseline
from sklearn.svm import SVC

def train_svm(X_train, y_train):
    model = SVC()
    model.fit(X_train, y_train)
    return model


## Phase 6: Attention-Based Model Implementation

Implement a neural network with attention for multi-label classification.


In [ ]:
# Attention-Based Model
class Attention(nn.Module):
    def __init__(self, input_dim):
        super(Attention, self).__init__()
        self.attention_layer = nn.Linear(input_dim, 1)
    def forward(self, x):
        weights = torch.softmax(self.attention_layer(x), dim=1)
        return torch.sum(x * weights, dim=1), weights

class AttentionModel(nn.Module):
    def __init__(self, input_dim, num_labels):
        super(AttentionModel, self).__init__()
        self.embedding = nn.Linear(input_dim, 128)
        self.attention = Attention(128)
        self.output = nn.Linear(128, num_labels)
    def forward(self, x):
        x = torch.relu(self.embedding(x))
        x, _ = self.attention(x)
        return torch.sigmoid(self.output(x))


## Phase 7: Evaluation and Error Analysis

1. Evaluate models using classification metrics.
2. Analyze misclassified segments using annotations.


In [ ]:
# Evaluation
def evaluate_model(y_true, y_pred, labels):
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.show()

    print(classification_report(y_true, y_pred))
